In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import streamlit as st

## [1] Data Loading

In [2]:
df = pd.read_csv('data/restaurant_sales_data.csv')

---

## [2] Data Understanding

In [3]:
df.shape

(17534, 9)

In [4]:
df.head(5)

,Order ID,Customer ID,Category,Item,Price,Quantity,Order Total,Order Date,Payment Method
0,ORD_705844,CUST_092,Side Dishes,Side Salad,3.0,1.0,3.0,2023-12-21,Credit Card
1,ORD_338528,CUST_021,Side Dishes,Mashed Potatoes,4.0,3.0,12.0,2023-05-19,Digital Wallet
2,ORD_443849,CUST_029,Main Dishes,Grilled Chicken,15.0,4.0,60.0,2023-09-27,Credit Card
3,ORD_630508,CUST_075,Drinks,NaN,NaN,2.0,5.0,2022-08-09,Credit Card
4,ORD_648269,CUST_031,Main Dishes,Pasta Alfredo,12.0,4.0,48.0,2022-05-15,Cash


In [5]:
df.info()
# There are missing values at these columns [Item, Price, Quantity, Order Total, Payment Method]
# Columns with their datatypes have to be modified [Order Date]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17534 entries, 0 to 17533
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Order ID        17534 non-null  object 
 1   Customer ID     17534 non-null  object 
 2   Category        17534 non-null  object 
 3   Item            15776 non-null  object 
 4   Price           16658 non-null  float64
 5   Quantity        17104 non-null  float64
 6   Order Total     17104 non-null  float64
 7   Order Date      17534 non-null  object 
 8   Payment Method  16452 non-null  object 
dtypes: float64(3), object(6)
memory usage: 1.2+ MB


In [6]:
df.describe()
# [Price, Order Total] have outliers

,Price,Quantity,Order Total
count,16658.000000,17104.000000,17104.000000
mean,6.586325,3.014149,19.914494
std,4.834652,1.414598,18.732549
min,1.000000,1.000000,1.000000
25%,3.000000,2.000000,7.500000
50%,5.000000,3.000000,15.000000
75%,7.000000,4.000000,25.000000
max,20.000000,5.000000,100.000000


---

## [3] Data Cleaning

### 1. Standardize the columns names

In [7]:
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')

In [8]:
df.head(5)

,order_id,customer_id,category,item,price,quantity,order_total,order_date,payment_method
0,ORD_705844,CUST_092,Side Dishes,Side Salad,3.0,1.0,3.0,2023-12-21,Credit Card
1,ORD_338528,CUST_021,Side Dishes,Mashed Potatoes,4.0,3.0,12.0,2023-05-19,Digital Wallet
2,ORD_443849,CUST_029,Main Dishes,Grilled Chicken,15.0,4.0,60.0,2023-09-27,Credit Card
3,ORD_630508,CUST_075,Drinks,NaN,NaN,2.0,5.0,2022-08-09,Credit Card
4,ORD_648269,CUST_031,Main Dishes,Pasta Alfredo,12.0,4.0,48.0,2022-05-15,Cash


### 2. Drop Unuseful columns for Analysis

In [9]:
df.drop(columns=['order_id'], inplace=True)

In [10]:
df.head(5)

,customer_id,category,item,price,quantity,order_total,order_date,payment_method
0,CUST_092,Side Dishes,Side Salad,3.0,1.0,3.0,2023-12-21,Credit Card
1,CUST_021,Side Dishes,Mashed Potatoes,4.0,3.0,12.0,2023-05-19,Digital Wallet
2,CUST_029,Main Dishes,Grilled Chicken,15.0,4.0,60.0,2023-09-27,Credit Card
3,CUST_075,Drinks,NaN,NaN,2.0,5.0,2022-08-09,Credit Card
4,CUST_031,Main Dishes,Pasta Alfredo,12.0,4.0,48.0,2022-05-15,Cash


### 3. Handle missing values

In [11]:
numerical = ['price', 'quantity', 'order_total']
categorical = ['customer_id', 'category', 'item', 'order_date']

In [12]:
df.isna().sum().sort_values(ascending=False)

item              1758
payment_method    1082
price              876
quantity           430
order_total        430
category             0
customer_id          0
order_date           0
dtype: int64

In [13]:
for i, col in enumerate(numerical):
    fig = px.box(df, x=col, title=f'{col} boxblot', color_discrete_sequence=[px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]])
    fig.show()

In [14]:
def calc_outliers_perct(col):
    col = col.dropna()

    q1 = col.quantile(0.25)
    q3 = col.quantile(0.75)
    iqr = q3-q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    return ((col < lower_bound) | (col > upper_bound)).mean() * 100

In [15]:
outliers_in_perct = {col : calc_outliers_perct(df[col]) for col in numerical}

In [16]:
outliers_in_perct

{'price': np.float64(13.873214071317086),
 'quantity': np.float64(0.0),
 'order_total': np.float64(8.237839101964454)}

In [17]:
for col in numerical:
    df[col] = df[col].fillna(df[col].median())

In [18]:
df.isna().sum().sort_values(ascending=False)

item              1758
payment_method    1082
category             0
customer_id          0
price                0
quantity             0
order_total          0
order_date           0
dtype: int64

In [19]:
payment_mode = df['payment_method'].mode()[0]
df['payment_method'] = df['payment_method'].fillna(payment_mode)

In [20]:
df['item'] = df.groupby('category')['item'].transform(
    lambda category: category.fillna(category.mode()[0] if not category.mode().empty else "Unknown")
)

In [21]:
df.isna().sum().sort_values(ascending=False)

customer_id       0
category          0
item              0
price             0
quantity          0
order_total       0
order_date        0
payment_method    0
dtype: int64

### 4. Fix DataTypes + Some of Feature Engineering

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17534 entries, 0 to 17533
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customer_id     17534 non-null  object 
 1   category        17534 non-null  object 
 2   item            17534 non-null  object 
 3   price           17534 non-null  float64
 4   quantity        17534 non-null  float64
 5   order_total     17534 non-null  float64
 6   order_date      17534 non-null  object 
 7   payment_method  17534 non-null  object 
dtypes: float64(3), object(5)
memory usage: 1.1+ MB


In [23]:
df.order_date.unique()

array(['2023-12-21', '2023-05-19', '2023-09-27', '2022-08-09',
       '2022-05-15', '2022-07-20', '2022-08-19', '2023-02-15',
       '2023-12-16', '2022-08-07', '2023-12-09', '2023-10-30',
       '2023-04-11', '2023-08-18', '2022-06-08', '2022-08-28',
       '2022-05-01', '2023-09-01', '2022-11-13', '2023-06-08',
       '2023-02-21', '2023-02-01', '2023-05-09', '2023-09-14',
       '2023-03-18', '2023-05-18', '2023-09-05', '2022-02-02',
       '2023-07-30', '2023-01-31', '2023-02-28', '2022-11-19',
       '2023-05-03', '2023-04-20', '2023-09-18', '2022-04-08',
       '2023-07-29', '2023-10-16', '2022-02-09', '2023-08-17',
       '2022-04-13', '2023-07-21', '2023-12-03', '2023-09-16',
       '2023-02-11', '2023-02-05', '2022-01-16', '2023-11-21',
       '2022-10-08', '2023-03-01', '2023-05-23', '2022-08-11',
       '2022-08-03', '2022-12-20', '2022-09-12', '2022-06-24',
       '2023-08-04', '2022-03-11', '2023-09-28', '2022-08-12',
       '2023-01-23', '2023-07-08', '2022-12-07', '2023-

In [24]:
df['order_date'] = pd.to_datetime(df['order_date'])

In [25]:
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_day'] = df['order_date'].dt.day
df['order_day_of_week'] = df['order_date'].dt.dayofweek
df['order_day_name'] = df['order_date'].dt.day_name()
df['is_weekend'] = df['order_day_of_week'].isin([5, 6]).astype(int)
df.drop(columns=['order_date'],inplace=True)

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17534 entries, 0 to 17533
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        17534 non-null  object 
 1   category           17534 non-null  object 
 2   item               17534 non-null  object 
 3   price              17534 non-null  float64
 4   quantity           17534 non-null  float64
 5   order_total        17534 non-null  float64
 6   payment_method     17534 non-null  object 
 7   order_year         17534 non-null  int32  
 8   order_month        17534 non-null  int32  
 9   order_day          17534 non-null  int32  
 10  order_day_of_week  17534 non-null  int32  
 11  order_day_name     17534 non-null  object 
 12  is_weekend         17534 non-null  int64  
dtypes: float64(3), int32(4), int64(1), object(5)
memory usage: 1.5+ MB


### 5. Remove Duplicates - if existed

In [27]:
df.duplicated().sum()

np.int64(6)

In [28]:
df.drop_duplicates(inplace=True)

In [29]:
df.duplicated().sum()

np.int64(0)

In [30]:
df.to_csv('data/cleaned_restaurant_sales_data.csv', index=False)

---

## [4] EDA

In [31]:
df = pd.read_csv('data/cleaned_restaurant_sales_data.csv')

In [32]:
df.head()

,customer_id,category,item,price,quantity,order_total,payment_method,order_year,order_month,order_day,order_day_of_week,order_day_name,is_weekend
0,CUST_092,Side Dishes,Side Salad,3.0,1.0,3.0,Credit Card,2023,12,21,3,Thursday,0
1,CUST_021,Side Dishes,Mashed Potatoes,4.0,3.0,12.0,Digital Wallet,2023,5,19,4,Friday,0
2,CUST_029,Main Dishes,Grilled Chicken,15.0,4.0,60.0,Credit Card,2023,9,27,2,Wednesday,0
3,CUST_075,Drinks,Water,5.0,2.0,5.0,Credit Card,2022,8,9,1,Tuesday,0
4,CUST_031,Main Dishes,Pasta Alfredo,12.0,4.0,48.0,Cash,2022,5,15,6,Sunday,1


In [33]:
numerical = ['price', 'quantity', 'order_total']
categorical = ['customer_id', 'category', 'item']

### Analysis: Distribution of Price

In [34]:
fig = px.histogram(df, x='price', title=f'Distribution of Item Prices', marginal='box', color_discrete_sequence=['#EF553B'])
fig.show()

**1. Central Tendency:**
* **Median ($5.0):**
    * The median price of an item ordered is $5.0; so the half of the items sold cost less than $5, and half cost more.
    * This is also the highest peak (mode) in the histogram, indicating it's the most common price point.

**2. Data Spreed (Box Plot Insites)**
* **Overall Range:** Item prices range from a minimum of `$1.0` up to a maximum of `$20.0`.
* **IQR:** The middle 50% of the item prices fall tightly between `$4.0` (Q1) and `$7.0` (Q3).

**3. Shape of Distribution:**
* **Right-Skewed:** The data is heavily right-skewed. The vast majority of sales are concentrated on lower-priced items (under $10), with a long "tail" extending to the right.

**4. Outliers Analysis:**
* The box plot's upper fence is at `$10.0`. Any prices above this are considered statistical outliers within this dataset.
* We can observe distinct outlier clusters at approximately `$12`, `$14`, `$15`, `$18`, and `$20`. These higher-priced items likely represent premium main dishes or large combination meals, which are ordered less frequently than cheaper typical items like sides or beverages.

### Analysis: Distribution of Item Quantities

In [35]:
fig = px.histogram(df, x='quantity', title=f'Distribution of Item Quantities', marginal='box', color_discrete_sequence=['#00CC96'])
fig.show()

**1. Central Tendency:**
* **Median (3):**
    * The median number of items ordered is exactly 3.
    * This sits perfectly in the middle of our data range, indicating a very balanced distribution.

**2. Data Spread (Box Plot Insights):**
* **Overall Range:** The quantity of items ordered in this dataset ranges strictly from a minimum of `1` to a maximum of `5`.
* **IQR:** The middle 50% of the data falls cleanly between `2` (Q1) and `4` (Q3). 

**3. Shape of Distribution:**
* **Uniform Distribution:** This histogram is flat. Customers are almost equally likely to order 1, 2, 3, 4, or 5 items in a single order. There is only a very marginal peak at a quantity of 3.

**4. Outliers Analysis:**
* **No Outliers:** The box plot reveals absolutely zero statistical outliers. The data is perfectly constrained between the minimum and maximum boundaries. This strongly suggests that there is a strict system limit enforcing a maximum of 5 identical items per order, or at least one per item per order.


### Analysis: Distribution of Total Price per Order

In [36]:
fig = px.histogram(df, x='order_total', title=f'Distribution of Total Price Per Order', marginal='box', color_discrete_sequence=['#AB63FA'])
fig.show()

**1. Central Tendency:**
* **Median ($15):**
    * The median total price of an order is `$15`.
    * Also it is the largest single peak (mode) in the histogram right around this value. This means half of all orders total less than $15, and half total more.

**2. Data Spread (Box Plot Insights):**
* **Overall Range:** Total order values span from a minimum of `$1` to a maximum of `$100`.
* **IQR:** The middle 50% of the revenue per order falls between `$8` (Q1) and `$24` (Q3). This `$8 to $24` range represents the "typical" order size for most customers.

**3. Shape of Distribution:**
* **Right-Skewed:** A long right tail pulls the distribution outwards to much higher order totals.

**4. Outliers Analysis:**
* **Upper Fence:** The box plot sets the upper fence at `$48`. Statistically, any single order totaling more than $48 is considered an outlier in this dataset.
* **Significant Outliers:** We see several outliers extending up to the maximum of `$100`. Noticeably, there is an artificial-looking spike exactly at `$60`, and another at the `$100` mark. These specific price points might correspond to set party bundles, other non most repeatative events.

### Analysis: order_month

In [37]:
monthly_counts = df['order_month'].value_counts().reset_index()

display(monthly_counts)

fig = px.bar(monthly_counts, x='order_month', y='count', title='Total Orders per Month', color='order_month', color_continuous_scale='Sunset')

fig.show()


,order_month,count
0,3,1554
1,8,1543
2,10,1538
3,1,1471
4,5,1456
5,9,1451
6,4,1449
7,12,1448
8,6,1448
9,7,1430


### Analysis: order_day_name

In [38]:
day_counts = df['order_day_name'].value_counts().reset_index()

display(day_counts)

fig = px.bar(day_counts, x='order_day_name', y='count', title='Total Orders by Day of the Week', color='order_day_name', color_discrete_sequence=px.colors.qualitative.Bold)
fig.show()


,order_day_name,count
0,Friday,2529
1,Saturday,2524
2,Monday,2524
3,Thursday,2515
4,Sunday,2483
5,Tuesday,2477
6,Wednesday,2476


### Analysis: is_weekend

In [39]:
weekend_counts = df['is_weekend'].value_counts().reset_index()

weekend_counts['Day Type'] = weekend_counts['is_weekend'].map({0: 'Weekday', 1: 'Weekend'})

display(weekend_counts)

fig = px.pie(weekend_counts, names='Day Type', values='count',                
             title='Percentage of Orders: Weekday vs. Weekend', color_discrete_sequence=px.colors.qualitative.D3)

fig.show()


,is_weekend,count,Day Type
0,0,12521,Weekday
1,1,5007,Weekend


### Analysis: Distribution of Customer ID

In [40]:
display(df[categorical].describe())

,customer_id,category,item
count,17528,17528,17528
unique,100,5,26
top,CUST_066,Main Dishes,Pasta Alfredo
freq,207,3550,1351


In [41]:
val_counts_customer = df['customer_id'].value_counts().reset_index()

display(val_counts_customer.head(10))

fig = px.bar(val_counts_customer.head(10), x='customer_id', y='count', title='Top 10 Customers with Most Orders', color='customer_id', color_discrete_sequence=px.colors.qualitative.Prism)
fig.show()

,customer_id,count
0,CUST_066,207
1,CUST_055,199
2,CUST_100,199
3,CUST_087,199
4,CUST_028,198
5,CUST_019,197
6,CUST_040,197
7,CUST_096,194
8,CUST_053,193
9,CUST_090,193


### Analysis: Distribution of Category

In [42]:
val_counts_category = df['category'].value_counts().reset_index()

display(val_counts_category)

fig = px.bar(val_counts_category, x='category', y='count', title='Frequency of Each Category', color='category', color_discrete_sequence=px.colors.qualitative.Safe)
fig.show()


,category,count
0,Main Dishes,3550
1,Starters,3527
2,Desserts,3495
3,Drinks,3487
4,Side Dishes,3469


### Analysis: Distribution of Items

In [43]:
val_counts_item = df['item'].value_counts().reset_index()

display(val_counts_item.head(10))

fig = px.bar(val_counts_item.head(10), x='item', y='count', title='Top 10 Most Frequent Items', color='item', color_discrete_sequence=px.colors.qualitative.Vivid)

fig.show()

,item,count
0,Pasta Alfredo,1351
1,Water,1334
2,Side Salad,1322
3,Ice Cream,1295
4,French Fries,1220
5,Grilled Chicken,822
6,Mashed Potatoes,798
7,Chocolate Cake,797
8,Coca Cola,756
9,Cheese Fries,686


---

## [5] Streamlit

In [49]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px
import numpy as np

st.set_page_config(page_title="Restaurant Sales Dashboard", layout="wide")

st.title("Restaurant Sales Dashboard")

# Load data
try:
    df = pd.read_csv('data/cleaned_restaurant_sales_data.csv')
except Exception as e:
    st.error(f"Error loading data: {e}")
    df = pd.DataFrame()

if df.empty:
    st.warning("No data available to display.")
else:
    df_filtered = df.copy()

    # --- Navigation ---
    page = st.sidebar.radio("Navigation", ["Dashboard", "Raw Data Preview", "Insights & Next Steps"])
    st.sidebar.markdown("---")

    # --- Sidebar Filters ---
    st.sidebar.header("Filters")

    if 'order_year' in df_filtered.columns:
        years = sorted(df_filtered['order_year'].unique().tolist())
        selected_years = st.sidebar.multiselect("Select Year", years, default=years)
        if selected_years:
            df_filtered = df_filtered[df_filtered['order_year'].isin(selected_years)]

    if 'order_month' in df_filtered.columns:
        months = sorted(df_filtered['order_month'].unique().tolist())
        selected_months = st.sidebar.multiselect("Select Month", months, default=months)
        if selected_months:
            df_filtered = df_filtered[df_filtered['order_month'].isin(selected_months)]

    if 'category' in df_filtered.columns:
        categories = df_filtered['category'].unique().tolist()
        selected_categories = st.sidebar.multiselect("Select Categories", categories, default=categories)
        if selected_categories:
            df_filtered = df_filtered[df_filtered['category'].isin(selected_categories)]

    if 'order_day_name' in df_filtered.columns:
        days = df_filtered['order_day_name'].unique().tolist()
        selected_days = st.sidebar.multiselect("Select Days of Week", days, default=days)
        if selected_days:
            df_filtered = df_filtered[df_filtered['order_day_name'].isin(selected_days)]


    # --- Display Page Content ---
    if page == "Dashboard":
        # Top level metrics
        st.markdown("### Key Metrics")
        col1, col2, col3, col4 = st.columns(4)
        
        with col1:
            total_revenue = df_filtered['order_total'].sum() if 'order_total' in df_filtered.columns else 0
            st.metric("Total Revenue", f"${total_revenue:,.2f}")
        
        with col2:
            total_orders = len(df_filtered)
            st.metric("Total Orders", f"{total_orders:,}")
            
        with col3:
            avg_order_value = df_filtered['order_total'].mean() if 'order_total' in df_filtered.columns else 0
            st.metric("Avg Order Value", f"${avg_order_value:,.2f}")
            
        with col4:
            total_items = df_filtered['quantity'].sum() if 'quantity' in df_filtered.columns else 0
            st.metric("Total Items Sold", f"{total_items:,.0f}")

        st.markdown("---")

        # Visualizations
        col_chart1, col_chart2 = st.columns(2)

        with col_chart1:
            if 'order_year' in df_filtered.columns and 'order_month' in df_filtered.columns and 'order_total' in df_filtered.columns:
                st.subheader("Monthly Revenue Trend")
                
                df_filtered['year_month'] = df_filtered['order_year'].astype(str) + '-' + df_filtered['order_month'].astype(str).str.zfill(2)
                
                monthly_sales = df_filtered.groupby('year_month')['order_total'].sum().reset_index()
                monthly_sales = monthly_sales.sort_values('year_month')
                
                fig1 = px.line(monthly_sales, x='year_month', y='order_total', 
                              labels={'year_month': 'Month', 'order_total': 'Total Sales ($)'},
                              markers=True)
                st.plotly_chart(fig1, use_container_width=True)

        with col_chart2:
            if 'category' in df_filtered.columns and 'order_total' in df_filtered.columns:
                st.subheader("Sales by Category")
                category_sales = df_filtered.groupby('category')['order_total'].sum().reset_index()
                fig2 = px.pie(category_sales, values='order_total', names='category', hole=0.4)
                st.plotly_chart(fig2, use_container_width=True)

        col_chart3, col_chart4 = st.columns(2)

        with col_chart3:
            if 'order_day_name' in df_filtered.columns and 'order_total' in df_filtered.columns:
                st.subheader("Revenue by Day of the Week")
                
                # Order the days logically instead of alphabetically
                day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
                day_sales = df_filtered.groupby('order_day_name')['order_total'].sum().reset_index()
                
                fig3 = px.bar(day_sales, x='order_day_name', y='order_total',
                             labels={'order_day_name': 'Day of Week', 'order_total': 'Total Sales ($)'},
                             color='order_day_name',
                             category_orders={'order_day_name': day_order})
                             
                st.plotly_chart(fig3, use_container_width=True)
                
            elif 'payment_method' in df_filtered.columns:
                st.subheader("Payment Methods")
                payment_counts = df_filtered['payment_method'].value_counts().reset_index()
                payment_counts.columns = ['payment_method', 'count']
                fig3 = px.bar(payment_counts, x='payment_method', y='count',
                             labels={'payment_method': 'Payment Method', 'count': 'Number of Orders'},
                             color='payment_method')
                st.plotly_chart(fig3, use_container_width=True)

        with col_chart4:
            if 'item' in df_filtered.columns and 'quantity' in df_filtered.columns:
                st.subheader("Top Selling Items")
                top_items = df_filtered.groupby('item')['quantity'].sum().reset_index().sort_values('quantity', ascending=False).head(10)
                fig4 = px.bar(top_items, x='quantity', y='item', orientation='h',
                             labels={'quantity': 'Quantity Sold', 'item': 'Item'},
                             color='quantity', color_continuous_scale='Blues')
                fig4.update_layout(yaxis={'categoryorder':'total ascending'})
                st.plotly_chart(fig4, use_container_width=True)

    elif page == "Raw Data Preview":
        st.subheader("Raw Data Preview")
        
        preview_df = df_filtered.drop(columns=['year_month']) if 'year_month' in df_filtered.columns else df_filtered
        
        st.dataframe(preview_df, use_container_width=True)

    elif page == "Insights & Next Steps":
        st.subheader("💡 Insights & Next Steps")
        
        st.markdown("""
### 1. Data Insights

#### Analysis: Distribution of Price
**1. Central Tendency:**
* **Median ($5.0):**
    * The median price of an item ordered is $5.0; so the half of the items sold cost less than $5, and half cost more.
    * This is also the highest peak (mode) in the histogram, indicating it's the most common price point.

**2. Data Spreed (Box Plot Insites)**
* **Overall Range:** Item prices range from a minimum of `$1.0` up to a maximum of `$20.0`.
* **IQR:** The middle 50% of the item prices fall tightly between `$4.0` (Q1) and `$7.0` (Q3).

**3. Shape of Distribution:**
* **Right-Skewed:** The data is heavily right-skewed. The vast majority of sales are concentrated on lower-priced items (under $10), with a long "tail" extending to the right.

**4. Outliers Analysis:**
* The box plot's upper fence is at `$10.0`. Any prices above this are considered statistical outliers within this dataset.
* We can observe distinct outlier clusters at approximately `$12`, `$14`, `$15`, `$18`, and `$20`. These higher-priced items likely represent premium main dishes or large combination meals, which are ordered less frequently than cheaper typical items like sides or beverages.

#### Analysis: Distribution of Item Quantities
**1. Central Tendency:**
* **Median (3):**
    * The median number of items ordered is exactly 3.
    * This sits perfectly in the middle of our data range, indicating a very balanced distribution.

**2. Data Spread (Box Plot Insights):**
* **Overall Range:** The quantity of items ordered in this dataset ranges strictly from a minimum of `1` to a maximum of `5`.
* **IQR:** The middle 50% of the data falls cleanly between `2` (Q1) and `4` (Q3). 

**3. Shape of Distribution:**
* **Uniform Distribution:** This histogram is flat. Customers are almost equally likely to order 1, 2, 3, 4, or 5 items in a single order. There is only a very marginal peak at a quantity of 3.

**4. Outliers Analysis:**
* **No Outliers:** The box plot reveals absolutely zero statistical outliers. The data is perfectly constrained between the minimum and maximum boundaries. This strongly suggests that there is a strict system limit enforcing a maximum of 5 identical items per order, or at least one per item per order.

#### Analysis: Distribution of Total Price per Order
**1. Central Tendency:**
* **Median ($15):**
    * The median total price of an order is `$15`.
    * Also it is the largest single peak (mode) in the histogram right around this value. This means half of all orders total less than $15, and half total more.

**2. Data Spread (Box Plot Insights):**
* **Overall Range:** Total order values span from a minimum of `$1` to a maximum of `$100`.
* **IQR:** The middle 50% of the revenue per order falls between `$8` (Q1) and `$24` (Q3). This `$8 to $24` range represents the "typical" order size for most customers.

**3. Shape of Distribution:**
* **Right-Skewed:** A long right tail pulls the distribution outwards to much higher order totals.

**4. Outliers Analysis:**
* **Upper Fence:** The box plot sets the upper fence at `$48`. Statistically, any single order totaling more than $48 is considered an outlier in this dataset.
* **Significant Outliers:** We see several outliers extending up to the maximum of `$100`. Noticeably, there is an artificial-looking spike exactly at `$60`, and another at the `$100` mark. These specific price points might correspond to set party bundles, other non most repeatative events.

---

### 2. Business Requirements & Next Steps
Based on our exploratory data analysis, the following business actions and requirements are recommended:

* **Review the 5-Item Quantity Limit:** The strict limit of 5 items per order is highly likely a technical constraint within the ordering system. **Action Required:** Discuss with the technical team to consider increasing this limit to capture larger group sizes.
* **Promote Premium Items:** High-priced items ($12-$20) exhibit outlier traits, meaning they are rarely ordered. **Action Required:** Introduce combo meals or promotional discounts targeting these premium dishes to boost their adoption.
* **Formalize Party Bundles:** The unusual spikes in order totals at exactly $60 and $100 suggest the occurrence of bulk custom orders. **Action Required:** Productize formal catering or "Party Bundles" priced around $60 and $100 to market explicitly to larger groups.
* **Marketing Focus on $5 Sweet Spot:** The most common item price is exactly $5.0. **Action Required:** Use this price point strategically in advertising to attract cost-conscious customers with a visible "Starting at $5" banner.
        """)


Overwriting app.py


In [50]:
!streamlit run app.py

^C
